In [60]:
from IPython.display import clear_output

from collections import defaultdict
import transformers

In [56]:
def avg_probability_per_text():
    pass

In [53]:
import json
from utils import compress_dict, decompress_dict
#Sample dictionary
#my_dict = {
#   "name": "Alice",
#   "age": 30,
#   "city": "Wonderland",
#   "hobbies": ["reading", "adventures", "tea parties"]
#}
#with open("./datasets/EFCAMDAT/cleaned_efcamdat.json") as inpf:
#   my_dict = json.load(inpf)

# Compress the dictionary
# compressed = compress_dict(my_dict)
filepath= "./datasets/CELVA/predictions_batch/tokenized_celva.csv.json.zlib_bert-base-uncased.json.zlib"
#with open(filepath,"wb") as outf:
#    outf.write(compressed)
def read_dataset_zlib_json(filepath):
    with open(filepath, "rb") as inpf:
        decompressed = decompress_dict(inpf.read())
    return decompressed
data = read_dataset_zlib_json(filepath)
#print("Compressed data:", compressed)
# Decompress the dictionary
# decompressed = decompress_dict(compressed)



In [ ]:
target_model = "bert-base-uncased"
model = transformers.AutoModelForMaskedLM.from_pretrained(target_model)
tokenizer= transformers.AutoTokenizer.from_pretrained(target_model)

## calculating stats

In [97]:
texts_stats = defaultdict(int)
inp = None
for text_id in data.keys():
    # print(len(data[text_id]['tokens']))
    first_token_has_no_prediction_of_target_model = data[text_id]['tokens'][0]['predictions']['models'].get(target_model,False)
    if not first_token_has_no_prediction_of_target_model:
        texts_stats["texts_missing_prediction"]+=1
        texts_stats["tokens_missing_prediction"]+=len(data[text_id]['tokens'])
        continue
    else:
        texts_stats["texts_has_prediction"] += 1
        texts_stats["tokens_has_prediction"]+=len(data[text_id]['tokens'])
        print(data[text_id]['tokens'][0]['predictions']['models'])
        #inp = input()
    if inp == "b":
        clear_output()
        break
    clear_output()

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [ ]:
texts_stats

## Annotating pooling

In [ ]:
# data[text_id]['tokens'][0]['token']
tokens_stats = defaultdict(lambda: defaultdict(int))
for text_id in data.keys():
    # print(len(data[text_id]['tokens']))
    first_token_has_no_prediction_of_target_model = data[text_id]['tokens'][0]['predictions']['models'].get(target_model,False)
    if not first_token_has_no_prediction_of_target_model:
        continue
    else:
        for token_dict in data[text_id]['tokens']:
            human_token_str=token_dict['token']['token_str']
            human_token_llm_tokens_idxs = tokenizer.encode(human_token_str)[1:-1]
            human_token_llm_tokens_idxs = [human_token_llm_tokens_idxs] if not isinstance(human_token_llm_tokens_idxs,list) else human_token_llm_tokens_idxs
            number_of_llm_tokens = len(human_token_llm_tokens_idxs)
            for token_prediction_dict in token['predictions']['models'][target_model]:
                llm_pred_str=tokenizer.decode(token_prediction_dict['token_vocab_idx'])
                if human_token_llm_tokens_idxs[0] == token_prediction_dict['token_vocab_idx']: # len(human_token_llm_tokens_idxs) == 1 and 
                    prob_score = token_prediction_dict['score']
                    clear_output()
                    print(prob_score)
                    break
                #else:
                #    print(human_token_str, llm_pred_str)
                #    input()
            #tokens_stats["llm_n_tokens"][number_of_llm_tokens]+=1
            #tokens_stats["token_freq"][token['token']['token_str']]+=1
            #print(llm_token_idx, number_of_llm_tokens)
            #tokenizer.decode([101,1999,102])
            #print(token['predictions']['models'][target_model])
    #inp = input()
    '''if inp == "b":
        clear_output()
        break'''
    clear_output()

9.678565220383462e-07


In [141]:
{k:v/sum(tokens_stats["llm_n_tokens"].values()) for k,v in tokens_stats["llm_n_tokens"].items()}

{}

In [131]:
{k:v/sum(tokens_stats["token_freq"].values()) for k,v in sorted(tokens_stats["token_freq"].items(),key=lambda tpl: tpl[1],reverse=True)}

{'.': 0.04699803500825487,
 'the': 0.04342013236336646,
 ',': 0.04043696087620552,
 'to': 0.031573087701435444,
 'a': 0.02475984755850965,
 'of': 0.023632237283458386,
 'I': 0.023322977081440106,
 'and': 0.023184999452847334,
 'in': 0.017979912360416597,
 'is': 0.014116538759819011,
 'that': 0.01047678407452695,
 'it': 0.010467268376003312,
 'this': 0.009468120031021177,
 'for': 0.00943481508618844,
 'my': 0.008954272310744651,
 'with': 0.007198625933133186,
 'was': 0.0070749218523258744,
 'have': 0.006494464242383873,
 'can': 0.006461159297551135,
 'are': 0.005790302551634559,
 'be': 0.00577127115458728,
 'we': 0.00548104234961628,
 'me': 0.00548104234961628,
 "'s": 0.005266939132834394,
 'on': 0.00501001527269613,
 'because': 0.0046151137839650965,
 'people': 0.004572293140608719,
 'do': 0.0045675352913469,
 'like': 0.00456277744208508,
 'i': 0.004434315512015948,
 'The': 0.00411078176221221,
 'or': 0.004010866927713996,
 'about': 0.003996593379928537,
 'an': 0.003825310806503028,
 '

In [107]:
# data[text_id]['tokens'][0]['token']['token_str'].lower()
print(tokenizer.encode(data[text_id]['tokens'][100]['token']['token_str']))
tokenizer.decode([101,1999,102])

[101, 1012, 102]


'[CLS] in [SEP]'

In [63]:
modules = [m for m in dir(transformers)]

In [67]:
[m for m in modules if("Auto" in m) ]

['AutoBackbone',
 'AutoConfig',
 'AutoFeatureExtractor',
 'AutoImageProcessor',
 'AutoModel',
 'AutoModelForAudioClassification',
 'AutoModelForAudioFrameClassification',
 'AutoModelForAudioXVector',
 'AutoModelForCTC',
 'AutoModelForCausalLM',
 'AutoModelForDepthEstimation',
 'AutoModelForDocumentQuestionAnswering',
 'AutoModelForImageClassification',
 'AutoModelForImageSegmentation',
 'AutoModelForImageToImage',
 'AutoModelForInstanceSegmentation',
 'AutoModelForKeypointDetection',
 'AutoModelForMaskGeneration',
 'AutoModelForMaskedImageModeling',
 'AutoModelForMaskedLM',
 'AutoModelForMultipleChoice',
 'AutoModelForNextSentencePrediction',
 'AutoModelForObjectDetection',
 'AutoModelForPreTraining',
 'AutoModelForQuestionAnswering',
 'AutoModelForSemanticSegmentation',
 'AutoModelForSeq2SeqLM',
 'AutoModelForSequenceClassification',
 'AutoModelForSpeechSeq2Seq',
 'AutoModelForTableQuestionAnswering',
 'AutoModelForTextEncoding',
 'AutoModelForTextToSpectrogram',
 'AutoModelForTextToW

In [69]:
modules

['ASTConfig',
 'ASTFeatureExtractor',
 'ASTForAudioClassification',
 'ASTModel',
 'ASTPreTrainedModel',
 'Adafactor',
 'AdamW',
 'AdamWeightDecay',
 'AdaptiveEmbedding',
 'AddedToken',
 'Agent',
 'AlbertConfig',
 'AlbertForMaskedLM',
 'AlbertForMultipleChoice',
 'AlbertForPreTraining',
 'AlbertForQuestionAnswering',
 'AlbertForSequenceClassification',
 'AlbertForTokenClassification',
 'AlbertModel',
 'AlbertPreTrainedModel',
 'AlbertTokenizer',
 'AlbertTokenizerFast',
 'AlignConfig',
 'AlignModel',
 'AlignPreTrainedModel',
 'AlignProcessor',
 'AlignTextConfig',
 'AlignTextModel',
 'AlignVisionConfig',
 'AlignVisionModel',
 'AltCLIPConfig',
 'AltCLIPModel',
 'AltCLIPPreTrainedModel',
 'AltCLIPProcessor',
 'AltCLIPTextConfig',
 'AltCLIPTextModel',
 'AltCLIPVisionConfig',
 'AltCLIPVisionModel',
 'AlternatingCodebooksLogitsProcessor',
 'AqlmConfig',
 'AudioClassificationPipeline',
 'AutoBackbone',
 'AutoConfig',
 'AutoFeatureExtractor',
 'AutoImageProcessor',
 'AutoModel',
 'AutoModelForAu